In [1]:
from __future__ import annotations

import re
import unicodedata

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

_PRE = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})


def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("­", "")
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s*|\n+")


def clauses(text: str):
    norm = normalize(text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]

    merged = []
    for i, c in enumerate(raw):
        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):
            merged.append(c + " " + raw[i + 1])
        else:
            merged.append(c)
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)
    return out


def _rx(*alts: str) -> re.Pattern:
    return re.compile("|".join(alts))


NEGATION = _rx(
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b", r"\babsence\b",
    r"\bno evidence\b", r"\bunremarkable\b", r"\bfree of\b", r"\bnone\b", r"\bnil\b",
    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b",
    r"\bpas de\b", r"\bsans\b", r"\baucune?\b", r"\babsence\b",
    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",
    r"\bkeine?\b", r"\bohne\b", r"\bnicht\b",
    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",
    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",
    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",
    r"\bδεν\b", r"\bχωρις\b", r"ουδεν",
    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",
)

NORMALITY = _rx(
    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",
    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",
    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",
    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt",
    r"φυσιολογικ", r"ακεραι",
    r"unauffallig", r"regelrecht", r"\bintakt\b",
    r"нормал", r"запазен", r"съхранен", r"\bбез особености\b",
    r"\bgaaf\b", r"\bnormaal\b",
)

UNCERTAIN = _rx(
    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",
    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",
    r"\bposible\b", r"sin criterios categoricos", r"\bdudos",
    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim",
    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b",
    r"πιθαν", r"υποπτ",
    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bV\.a\.\b",
    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",
    r"\bmogelijk\b", r"\bverdacht\b",
)

TEAR = _rx(
    r"\btear", r"\btorn\b", r"\brupture", r"\bdisruption\b", r"discontinuit",
    r"\bavuls",
    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",
    r"\bdechirure", r"\bdechire",
    r"\bscheur", r"\bruptuur", r"gescheurd",
    r"riss(bildung|e|es)?\b", r"einriss", r"\bruptur", r"zerreiss", r"\blasion",
    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi", r"\brupturu\b",
    r"\bpuknuce", r"\bruptur", r"\bprekid\b", r"\bpukotin",
    r"ρηξη", r"ρηξις", r"ρηγμα",
    r"руптура", r"разкъсв", r"разрив", r"скъсв",
)

DEGEN = _rx(
    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",
    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλ", r"дегенерат",
    r"\bμυξοειδ", r"\bμυξωδ",
    r"\bmuco ?ide\b", r"aufgefasert",
)

INJURY = _rx(
    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",
    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",
    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",
    r"aumento de senal", r"alteracion de senal", r"cambio de senal",
    r"\bsignalanhebung", r"\bsignalalteration", r"verhoogd signaal", r"sinyal artis",
    r"αυξημενο σημα", r"повишен сигнал",
    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",
    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",
    r"\bpartiel", r"\bpartiell",
)

ANAT = {
    "ACL": _rx(
        r"anterior cruciate", r"\bacl\b",
        r"cruzado anterior", r"\blca\b",
        r"croise anterieur",
        r"voorste kruisband", r"\bvkb\b",
        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",
        r"on capraz", r"\bocb\b",
        r"prednji krizni", r"prednjeg krizn",
        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",
        r"\bχιαστ\w*",
        r"предна кръстна", r"предната кръстна",
        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",
        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",
        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",
    ),
    "MCL": _rx(
        r"medial collateral", r"\bmcl\b", r"tibial collateral",
        r"colateral medial", r"colateral interno", r"\blcm\b",
        r"collateral medial", r"collateral interne",
        r"mediale collaterale", r"binnenband", r"\b(mediale|laterale) banden\b",
        r"\bcollaterale banden\b",
        r"innenband", r"mediales? kollateral",
        r"\bic yan bag", r"medial kollateral", r"\biyb\b",
        r"medijalni kolateraln", r"medijalnog kolateraln",
        r"εσω πλαγι", r"εσωτερικο πλαγι", r"\bπλαγι\w* συνδεσμ", r"\bπλαγιοι\b",
        r"медиален колатерал", r"вътрешна странична", r"\bколатерал\w*",
        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",
        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",
        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",
        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",
        r"колатерални връзки", r"страничните връзки",
    ),
    "Medial Meniscus": _rx(
        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",
        r"menisco medial", r"menisco interno",
        r"menisque medial", r"menisque interne",
        r"mediale meniscus", r"binnenmeniscus",
        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",
        r"medyal menisk", r"\bic menisk",
        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",
        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",
        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",
    ),
    "Lateral Meniscus": _rx(
        r"lateral meniscus", r"lateral menisc",
        r"menisco lateral", r"menisco externo",
        r"menisque lateral", r"menisque externe",
        r"laterale meniscus", r"buitenmeniscus",
        r"aussenmeniskus", r"lateralen? meniskus",
        r"lateral menisk", r"\bdis menisk",
        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",
        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",
        r"латералния менискус", r"латерален менискус", r"външния менискус",
    ),
}

OA_EVIDENCE = _rx(
    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",
    r"chondropath", r"chondromalac", r"condropat", r"condromalac",
    r"cartilage loss", r"cartilage thinning", r"chondral (loss|defect|ulcer|thinning)",
    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten",
    r"joint space narrowing", r"pinzamiento articular",
    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral",
    r"kraakbeen(lijden|verlies)", r"gonartrose", r"artrose",
    r"knorpel(verlust|schaden|defekt)", r"arthrose", r"gonarthrose",
    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit",
    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ",
    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου",
    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,30}(изтън|увред|дефект)",
    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",
    r"icrs grade", r"outerbridge",
)

COMPARTMENT = {
    "Medial OA": _rx(
        r"medial (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial medial", r"femorotibial interno",
        r"mediaal femorotibiaal", r"mediale femorotibial",
        r"medial femorotibial", r"medialen kompartiment", r"innere[sn]? kompartiment",
        r"medyal femorotibial", r"ic kompartman", r"medyal kompartman",
        r"medijaln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εσω διαμερισμα", r"εσω κνημιαι", r"εσω μηριαι",
        r"медиалн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"medial (femoral|tibial) (condyle|plateau)", r"condilo femoral medial",
        r"medialen? (femurkondyl|tibiaplateau)", r"mediale femorale condyl",
    ),
    "Lateral OA": _rx(
        r"lateral (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial lateral", r"femorotibial externo",
        r"lateraal femorotibiaal", r"laterale femorotibial",
        r"lateral femorotibial", r"lateralen kompartiment", r"aussere[sn]? kompartiment",
        r"lateral femorotibial", r"dis kompartman", r"lateral kompartman",
        r"lateraln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εξω διαμερισμα", r"εξω κνημιαι", r"εξω μηριαι",
        r"латералн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"lateral (femoral|tibial) (condyle|plateau)", r"condilo femoral lateral",
        r"lateralen? (femurkondyl|tibiaplateau)", r"laterale femorale condyl",
    ),
    "PF OA": _rx(
        r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",
        r"retropatellar", r"retrorotulian", r"\btrochlea", r"\btroclea", r"\btroklea",
        r"\bpatella\b", r"\bpatellar\b", r"\brotulian", r"\brotula\b", r"\bpatele\b",
        r"\bpatellae?\b", r"patellofemoraal", r"femoropatellair",
        r"επιγονατιδ", r"μηροεπιγονατιδ", r"τροχιλ",
        r"пател", r"феморопател", r"тролх",
        r"anterior compartment", r"compartimento anterior", r"prednj[^ ]* odjeljk",
    ),
}

DIRECT = {
    "Effusion": _rx(
        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",
        r"derrame articular", r"\bderrame\b", r"liquido articular",
        r"epanchement",
        r"gewrichtsvocht", r"\bvocht\b", r"\bhydrops\b", r"gewrichtseffusie",
        r"gelenkerguss", r"\berguss\b", r"gelenksergu",
        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi",
        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",
        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",
        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",
        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",
        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",
    ),
    "Synovitis": _rx(
        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",
        r"synovitis", r"synoviale? (verdikking|proliferatie)",
        r"synovialitis", r"synovialis(verdickung|proliferation)",
        r"sinovijalitis", r"sinovitis", r"zadebljanje sinovij",
        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",
        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",
        r"verdikkingen van (het )?synovium", r"pannus",
    ),
    "Baker's": _rx(
        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",
        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",
        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist",
        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",
        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста",
        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",
    ),
    "Contusion": _rx(
        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",
        r"\bkontuz", r"medular bone o?edema", r"marrow o?edema",
        r"contusion osea", r"edema oseo", r"edema de medula osea",
        r"oedeme osseux", r"contusion osseuse",
        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",
        r"knochenmarkodem", r"knochenodem", r"kontusion", r"bone bruise",
        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi",
        r"kostani edem", r"edem kosti", r"kontuzij",
        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα",
        r"костномозъчен едем", r"костен едем", r"контузионен",
    ),
    "Fracture": _rx(
        r"\bfractur", r"\bfract\b",
        r"\bfractura", r"\bfracturas\b",
        r"\bfractuur", r"\bbreuk\b",
        r"\bfraktur", r"\bbruch\b",
        r"\bkirik\b", r"\bkirigi\b", r"\bkirik\b",
        r"\bfraktur", r"\bprijelom", r"impresijsk[^ ]* fraktur",
        r"καταγμα", r"καταγματ",
        r"фрактур", r"счупван", r"фисур",
        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",
        r"subchondral fracture", r"subkondral kiri",
    ),
}

DECOY = {
    "Fracture": _rx(r"microfractur", r"\bfracture (risk|prophyla)"),
    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"ganglion"),
}

PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_TARGETS = {"Medial OA", "Lateral OA", "PF OA"}

STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")
STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",
                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",
                    r"\bacl\b", r"\bpcl\b", r"\blca\b", r"\blcp\b", r"\bvkb\b",
                    r"\bhkb\b", r"\bocb\b", r"\bacb\b")
STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",
                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",
                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",
                      r"innenband\w*", r"aussenband\w*", r"binnenband\w*",
                      r"\bmcl\b", r"\blcl\b", r"\blcm\b", r"\biyb\b")

SIDE_MEDIAL = _rx(r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",
                  r"\bmediale\w*", r"\bintern[oa]\w*", r"\binterne\w*", r"\binnen\w*",
                  r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",
                  r"\bмедиал\w*", r"\bвътреш\w*", r"\btibial collateral\b",
                  r"\bbinnen\w*", r"\bmediaal\b")
SIDE_LATERAL = _rx(r"\blateral\w*", r"\bextern[oa]\w*", r"\bexterne\w*", r"\bdis\b",
                   r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",
                   r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*",
                   r"\bfibular collateral\b", r"\bvanjsk\w*")
SIDE_ANTERIOR = _rx(r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*",
                    r"\bvorder\w*", r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*",
                    r"\banteriyor\w*", r"\bavant\b", r"\bant[eé]rieur\w*")

SIDE_POSTERIOR = _rx(r"\bposterior\w*", r"\bpost[eé]rieur\w*", r"\bposteriore\w*",
                     r"\bhinter\w*", r"\bachterste\b", r"\barka\b", r"\bstraznj\w*",
                     r"\bzadnj\w*", r"\bοπισθι\w*", r"\bзадн\w*", r"\bpostero\w*")

STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",
                    r"kiri[kgğ]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",
                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",
                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")

STEM_OA_COMPARTMENT = _rx(r"compartment\w*", r"compartimento\w*", r"compartiment\w*",
                          r"kompartman\w*", r"kompartiment\w*", r"odjelj\w*",
                          r"διαμερισμα\w*", r"компартм\w*", r"\bотдел\w*",
                          r"femorotibial\w*", r"femorotibiaal\w*", r"tibiofemoral\w*",
                          r"femoro tibial\w*", r"κνημιαι\w*", r"μηριαι\w*",
                          r"femoral condyl\w*", r"tibial plateau\w*",
                          r"condilo femoral", r"platillo tibial", r"tibiaplateau\w*",
                          r"femurkondyl\w*", r"femoralne? kondil\w*",
                          r"tibijaln\w* plato", r"femoral kondil\w*",
                          r"tibia plato", r"tibyal plato")


def _distance(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    best = None
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        for q in qual_rx.finditer(clause[lo:hi]):
            qs, qe = lo + q.start(), lo + q.end()
            d = 0 if qs < m.end() and qe > m.start() else\
                min(abs(m.start() - qe), abs(qs - m.end()))
            best = d if best is None else min(best, d)
    return best


def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    return _distance(clause, stem_rx, qual_rx, window) is not None


STEM_RULES = {
    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),
    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),
    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),
    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),
    "Medial OA": (STEM_OA_COMPARTMENT, SIDE_MEDIAL),
    "Lateral OA": (STEM_OA_COMPARTMENT, SIDE_LATERAL),
}

SEV_LOW = _rx(
    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",
    r"\btiny\b", r"\bscant\b", r"\bmimimal\b", r"\bdiscrete\b", r"\bfocal\b",
    r"\bleve\b", r"\bminim", r"\bpeque", r"\bligero\b", r"\bescaso\b", r"\bdiscreto\b",
    r"\bhafif\b", r"\bminimal\b", r"\baz miktarda\b", r"\bsilik\b",
    r"\bmanja\b", r"\bmanji\b", r"\bblago\b", r"\bdiskretn", r"\bmalo\b",
    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",
    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b",
    r"\bηπι", r"\bμικρ", r"\bελαχιστ",
    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",
)

SEV_HIGH = _rx(
    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",
    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",
    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",
    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b",
    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",
    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",
    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b",
    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ",
    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен",
)

GLOBAL_OA = _rx(
    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",
    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose",
    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",
    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",
    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος",
    r"артроза на колянната", r"гонартроз",
    r"degenerative joint disease", r"\bdjd\b",
)

DEGENERATIVE_MARROW = _rx(
    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",
    r"υποχονδρι", r"субхондрал", r"subchondrale?",
    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",
)

TRAUMA = _rx(
    r"\bbruise\b", r"\bcontusion", r"\bkontuz", r"\bcontusion osea\b",
    r"\btrauma", r"\bimpaction\b", r"\bpivot shift\b", r"\bkissing\b",
    r"\bacute\b", r"\bagudo\b", r"\bakut", r"\bpivot kaymasi\b",
    r"\bcontusion osseuse\b", r"\bbone bruise\b", r"\bbotcontusie\b",
    r"\bконтузион", r"\bμωλωπ", r"\bkontuzij",
)


def _polarity(clause: str, anchor_end: int) -> str:
    if UNCERTAIN.search(clause):
        return "uncertain"
    if NEGATION.search(clause):
        return "negative"
    if NORMALITY.search(clause):
        if TEAR.search(clause) or re.search(r"\bgrade [34]\b", clause):
            return "positive"
        return "negative"
    return "positive"


class _Matcher:
    def __init__(self, phrase_rx, stem=None, side=None, window=55, contrary=None):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window
        self.contrary = contrary

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None and not self._wrong_side(clause):
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None

    def _wrong_side(self, clause):
        if self.contrary is None or self.stem is None:
            return False
        other = _distance(clause, self.stem, self.contrary, self.window)
        if other is None:
            return False
        own = _distance(clause, self.stem, self.side, self.window)
        return own is None or other < own


CONTRARY = {"ACL": SIDE_POSTERIOR, "MCL": SIDE_LATERAL}

ANAT_MATCH = {
    tgt: _Matcher(ANAT[tgt], *STEM_RULES[tgt], contrary=CONTRARY.get(tgt))
    for tgt in PAIRED
}
COMPARTMENT_MATCH = {
    "Medial OA": _Matcher(COMPARTMENT["Medial OA"], *STEM_RULES["Medial OA"]),
    "Lateral OA": _Matcher(COMPARTMENT["Lateral OA"], *STEM_RULES["Lateral OA"]),
    "PF OA": _Matcher(COMPARTMENT["PF OA"]),
}
DIRECT_MATCH = {
    tgt: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if tgt == "Fracture" else rx)
    for tgt, rx in DIRECT.items()
}


def _severity(clause: str) -> float:
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and not low:
        return 1.0
    if low and not high:
        return 0.45
    return 0.75


def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None,
                   context_bonus=None):
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and not path_rx.search(c):
            if NORMALITY.search(c) and not NEGATION.search(c):
                n_neg += 1
            continue
        pol = _polarity(c, m.end())
        if pol == "positive":
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.30)

    if n_pos or n_unc:
        score = min(0.95, 0.50 + 0.42 * best + 0.03 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.20 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = 0.28, 0.05
    return score, conf, n_pos, n_neg


EFF_STRONG = _rx(
    r"\bmoderate\b", r"\blarge\b", r"\bsignificant\b", r"\btense\b",
    r"\bhemarthrosis\b", r"\bhaemarthrosis\b", r"\bhemorrhagic\b", r"\bhaemorrhagic\b",
    r"\bhydrops\b", r"\bextensive\b", r"\bmassive\b", r"\bmarked\b",
    r"\bmoderad\w*\b", r"\bimportante\b", r"\bhemartrosis\b", r"\bhemorragic\w*\b",
    r"\bmatige?\b", r"\buitgebreid\b", r"\bopzetting\b",
    r"\byaygin\b", r"\bbelirgin\b", r"\bbol\b",
    r"\bopsezan\b", r"\bznacajn\w*\b", r"\bumjeren\w*\b", r"\bizrazit\w*\b",
    r"хеморагич\w*", r"значителн\w*", r"обилен\w*",
    r"εκτεταμεν\w*", r"μετρι\w*", r"σοβαρ\w*",
)

TRAUMA_WORD = _rx(
    r"\bcontusion\b", r"\bbone bruise\b", r"\bbruise\b", r"\bimpaction\b",
    r"\bkissing\b", r"\bacute\b",
    r"\bcontusi\w*\b", r"\bcontuz\w*\b", r"\bkontuz\w*\b",
    r"\bbotcontusie\b", r"\bbotoedeem\b", r"\bkontusion\w*\b",
    r"контузион\w*", r"травматич\w*",
    r"μωλωπ\w*",
    r"\bakut\w*\b", r"\btraumatisk\w*\b", r"\btraumatic\b", r"\bagudo\b",
)

DEGEN_EXCLUDE = _rx(
    r"\bsubchondral\b", r"\bsubcondral\b", r"\bsubkondral\b",
    r"\breactive\b", r"\breactivo\b",
    r"\bdegenerativ\w*\b", r"\bosteoarthrit\w*\b", r"\barthros\w*\b",
    r"\bchondropath\w*\b", r"\bcartilage (loss|defect|thinning)\b",
    r"\bcondropat\w*\b", r"\bosteophyt\w*\b",
    r"\binsufficiency fracture\b", r"\bstress fracture\b", r"\bsonk\b",
    r"\berosiv\w*\b", r"\bchondrosis\b", r"\bcondrosis\b",
    r"\bmarginal spurring\b", r"\bosteofit\w*\b",
)


def _synovitis_v4(cls):
    hits = [c for c in cls if DIRECT_MATCH["Synovitis"].search(c) and _polarity(c, 0) == "positive"]
    if hits:
        return 1.0
    eff_hits = [c for c in cls if DIRECT_MATCH["Effusion"].search(c) and _polarity(c, 0) == "positive"]
    if any(EFF_STRONG.search(c) for c in eff_hits):
        return 1.0
    return 0.0


def _contusion_v4(cls):
    hits = [c for c in cls if DIRECT_MATCH["Contusion"].search(c) and _polarity(c, 0) == "positive"]
    for c in hits:
        if TRAUMA_WORD.search(c) and not DEGEN_EXCLUDE.search(c):
            return 1.0
    return 0.0


def extract(report: str) -> dict:
    cls = clauses(report)
    out = {}
    path_paired = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)

    for tgt in TARGETS:
        if tgt in ("Synovitis", "Contusion"):
            continue
        if tgt in PAIRED:
            s, c, npos, nneg = _score_clauses(cls, ANAT_MATCH[tgt], path_paired)
        elif tgt in OA_TARGETS:
            s, c, npos, nneg = _score_clauses(cls, COMPARTMENT_MATCH[tgt], OA_EVIDENCE)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    g_hits = [c for c in cls if GLOBAL_OA.search(c) and _polarity(c, 0) == "positive"]
    if g_hits:
        gscore = 0.50 + 0.42 * max(_severity(c) for c in g_hits)
        for tgt in OA_TARGETS:
            if out[tgt + "__npos"] == 0 and out[tgt + "__nneg"] == 0:
                out[tgt] = max(out[tgt], gscore * 0.92)
                out[tgt + "__conf"] = max(out[tgt + "__conf"], 0.4)

    out["Synovitis"] = _synovitis_v4(cls)
    out["Contusion"] = _contusion_v4(cls)

    return out


if __name__ == "__main__":
    import pandas as pd

    TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
    train = pd.read_csv(TRAIN_CSV)
    gold = train.dropna(subset=TARGETS)
    unlabeled = train[train[TARGETS].isna().any(axis=1)]
    print(f"gold-labeled studies: {len(gold)}")
    print(f"unlabeled studies to process: {len(unlabeled)}")


    correct = {t: 0 for t in TARGETS}
    total = {t: 0 for t in TARGETS}
    for _, row in gold.iterrows():
        pred = extract(row["Report"])
        for t in TARGETS:
            total[t] += 1
            if int(pred[t] >= 0.5) == int(row[t]):
                correct[t] += 1

    print("\nper-label accuracy on the gold set:")
    for t in TARGETS:
        print(f"{t:20s} {correct[t]}/{total[t]}  {correct[t]/total[t]:.1%}")
    overall = sum(correct.values()) / sum(total.values())
    print(f"\noverall label accuracy: {overall:.1%}")


    silver_rows = []
    for _, row in unlabeled.iterrows():
        pred = extract(row["Report"])
        silver_rows.append({
            "StudyInstanceUID": row["StudyInstanceUID"],
            **{t: int(pred[t] >= 0.5) for t in TARGETS},
        })
    silver = pd.DataFrame(silver_rows)
    silver["label_source"] = "silver_regex_v4"
    silver.to_csv("report_labels_silver_regex.csv", index=False)
    print(f"\nlabeled {len(silver)} previously-unlabeled studies -> report_labels_silver_regex.csv")


    gold_out = gold[["StudyInstanceUID"] + TARGETS].copy()
    gold_out["label_source"] = "gold"
    full_labels = pd.concat(
        [gold_out, silver[["StudyInstanceUID"] + TARGETS + ["label_source"]]],
        ignore_index=True,
    )
    full_labels.to_csv("report_labels_v1_regex.csv", index=False)
    print(full_labels["label_source"].value_counts())
    print(f"total labeled studies: {len(full_labels)} / {len(train)}")

gold-labeled studies: 58
unlabeled studies to process: 4349

per-label accuracy on the gold set:
ACL                  52/58  89.7%
MCL                  51/58  87.9%
Medial Meniscus      45/58  77.6%
Lateral Meniscus     50/58  86.2%
Medial OA            49/58  84.5%
Lateral OA           46/58  79.3%
PF OA                43/58  74.1%
Effusion             41/58  70.7%
Synovitis            42/58  72.4%
Baker's              51/58  87.9%
Contusion            45/58  77.6%
Fracture             47/58  81.0%

overall label accuracy: 80.7%

labeled 4349 previously-unlabeled studies -> report_labels_silver_regex.csv
label_source
silver_regex_v4    4349
gold                 58
Name: count, dtype: int64
total labeled studies: 4407 / 4407


In [2]:
from __future__ import annotations

import gc
import json
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn.functional as F

T0 = time.time()


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


CROP_MM = 130.0
CACHE_IMG = 336
CACHE_SLICES = 3
SLICE_BAND = (0.20, 0.80)
LAT_MIN_OFFSET_MM = 20.0

HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
FLUSH_EVERY = 500

OUT_DIR = Path("preprocessed")
ORDER_CACHE_PATH = OUT_DIR / "order_cache.json"

SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")

HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            "ImagePositionPatient", "ImageOrientationPatient"]

ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

DECODE_FAILED = []


def find_root():
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError(
        f"competition mount not found (cwd {Path.cwd()}); expected a directory holding "
        f"test.csv and test_series/")


def available_gb():
    try:
        with open("/proc/meminfo") as fh:
            info = {k.strip(): v for k, v in
                    (l.split(":", 1) for l in fh if ":" in l)}
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return 16.0


def plan_cache(n_study, cache_fraction=0.45, budget_max_gb=24.0):
    avail = available_gb()
    budget = min(avail * cache_fraction, budget_max_gb)
    per_slice = n_study * N_SLOT * CACHE_IMG * CACHE_IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    slices = max(1, min(CACHE_SLICES, afford))
    log(f"memory: {avail:.1f} GB available, {budget:.1f} GB budgeted for the cache "
        f"-> {slices} slices/slot" + (f" (wanted {CACHE_SLICES})" if slices < CACHE_SLICES else ""))
    return slices


def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series, "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(root, split):
    base = root / f"{split}_series"
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out


def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def order_slices(rec):
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def lat_of(h, tag=""):
    geo = side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d


def normalise_laterality(img, plane, lat):
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


def read_slot(rec, n_slice, out_size):
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None

    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes, errors = [], []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception as exc:
            a = None
            errors.append(f"{type(exc).__name__}: {str(exc)[:120]}")
        planes.append(a)

    got = [k for k, p in enumerate(planes) if p is not None]
    if not got:
        DECODE_FAILED.append({"series": rec.get("SeriesInstanceUID", d), "errors": errors})
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append({"series": rec.get("SeriesInstanceUID", d), "errors": errors})
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


def _atomic_save_npy(path, arr):
    path = Path(path)
    tmp = path.with_name(path.stem + ".tmp.npy")
    np.save(tmp, arr)
    os.replace(tmp, path)


def open_or_create_memmap(path, shape, dtype=np.uint8):
    path = Path(path)
    nbytes = int(np.prod(shape)) * np.dtype(dtype).itemsize
    if path.is_file() and path.stat().st_size == nbytes:
        log(f"resuming from existing {path.name} ({nbytes / 1024 ** 3:.2f} GB)")
        return np.memmap(path, dtype=dtype, mode="r+", shape=shape), True
    path.parent.mkdir(parents=True, exist_ok=True)
    log(f"creating new {path.name} ({nbytes / 1024 ** 3:.2f} GB)")
    return np.memmap(path, dtype=dtype, mode="w+", shape=shape), False


def build_cache(slot_map, cache_slices, img_size, tag, out_dir,
                order_budget_s=ORDER_BUDGET_S, order_cache_path=ORDER_CACHE_PATH,
                flush_every=FLUSH_EVERY):
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    shape = (len(studies), N_SLOT, cache_slices, img_size, img_size)

    cache_path = Path(out_dir) / f"{tag}_cache.dat"
    mask_path = Path(out_dir) / f"{tag}_mask.npy"

    cache, resumed = open_or_create_memmap(cache_path, shape)
    if resumed and mask_path.is_file():
        prev_mask = np.load(mask_path)
        if prev_mask.shape == (len(studies), N_SLOT):
            mask = prev_mask.astype(np.float32)
        else:
            log(f"{tag}: mask shape mismatch on resume ({prev_mask.shape} vs "
                f"{(len(studies), N_SLOT)}); restarting mask and cache together")
            cache, resumed = open_or_create_memmap(cache_path, shape)
            cache[:] = 0
            mask = np.zeros((len(studies), N_SLOT), np.float32)
            resumed = False
    else:
        mask = np.zeros((len(studies), N_SLOT), np.float32)

    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB"
        + (" (resumed)" if resumed else ""))

    all_jobs = [(st, k, plane, slot_map[st][name])
                for st in studies
                for k, (name, plane, _, _) in enumerate(SLOTS)
                if name in slot_map[st]]
    n_job_total = len(all_jobs)

    jobs = [j for j in all_jobs if mask[sidx[j[0]], j[1]] < 0.5]
    if resumed:
        log(f"{tag}: {n_job_total - len(jobs)}/{n_job_total} slots already cached, "
            f"{len(jobs)} remaining")

    seen = {}
    if order_cache_path and Path(order_cache_path).is_file():
        try:
            seen = json.loads(Path(order_cache_path).read_text())
        except (OSError, ValueError):
            seen = {}

    t_ord = time.time()
    ok = 0
    hit = 0
    for _, _, _, rec in jobs:
        e = seen.get(rec["SeriesInstanceUID"])
        if e and len(e["files"]) == len(rec["files"]):
            rec["ordered"] = e["files"]
            ok += int(e["good"])
            hit += 1
    to_order = [j for j in jobs if "ordered" not in j[3]]
    log(f"{tag}: {hit} slot-series ordered from cache, {len(to_order)} to read "
        f"({sum(len(j[3]['files']) for j in to_order)} slice headers)")

    done = 0
    last_log = time.time()
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        futures = {pool.submit(order_slices, rec): rec for _, _, _, rec in to_order}
        for fut in as_completed(futures):
            rec = futures[fut]
            files, good = fut.result()
            rec["ordered"] = files
            ok += int(good)
            done += 1
            if order_cache_path:
                seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            now = time.time()
            if now - last_log > 15:
                rate = done / max(now - t_ord, 1e-6)
                remain = len(to_order) - done
                eta_s = remain / max(rate, 1e-6)
                log(f"  {tag} ordering {done}/{len(to_order)} "
                    f"({rate:.0f}/s, ~{eta_s / 60:.0f}min remaining)")
                last_log = now
            if now - t_ord > order_budget_s:
                log(f"{tag}: ordering budget spent at {done}/{len(to_order)}; "
                    f"rest keep file order")
                for f in futures:
                    f.cancel()
                break
    if order_cache_path and done:
        tmp = Path(order_cache_path).with_suffix(".tmp")
        tmp.write_text(json.dumps(seen))
        tmp.replace(order_cache_path)
    log(f"{tag}: ordered {ok}/{len(jobs)} by geometry ({len(jobs) - ok} kept arbitrary) "
        f"in {time.time() - t_ord:.0f}s")

    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)
    done = 0
    since_flush = 0


    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        futures = {pool.submit(read_slot, rec, cache_slices, img_size): (st, k, plane)
                   for st, k, plane, rec in jobs}
        for fut in as_completed(futures):
            st, k, plane = futures[fut]
            done += 1
            since_flush += 1
            try:
                img = fut.result()
            except Exception as exc:
                DECODE_FAILED.append({"series": st, "errors": [f"{type(exc).__name__}: {str(exc)[:120]}"]})
                img = None
            if img is not None:
                cache[sidx[st], k] = normalise_laterality(img, plane, _CURRENT_LAT.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if since_flush >= flush_every:
                cache.flush()
                _atomic_save_npy(mask_path, mask)
                since_flush = 0
            if done % 4096 < 1:
                log(f"  {tag} {done}/{len(jobs)}")
    cache.flush()
    _atomic_save_npy(mask_path, mask)
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{n_job_total} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask


def coverage_report(tag, studies, cache, mask, slot_map, lat_map):
    n_study = len(studies)
    log(f"=== {tag} coverage report ===")
    log(f"studies: {n_study}")
    log(f"overall slot fill rate: {mask.mean():.1%}")
    for k, (name, plane, fluid, fs) in enumerate(SLOTS):
        log(f"  {name:16s} {int(mask[:, k].sum())}/{n_study} ({mask[:, k].mean():.1%})")

    n_lat_resolved = sum(1 for st in studies if lat_map.get(st) is not None)
    log(f"laterality resolved: {n_lat_resolved}/{n_study} "
        f"({n_lat_resolved / max(n_study, 1):.1%})")

    n_chosen = n_unscaled = 0
    for st in studies:
        for name, row in slot_map.get(st, {}).items():
            n_chosen += 1
            px = row.get("px")
            if px is None or not np.isfinite(px) or px <= 0:
                n_unscaled += 1
    if n_chosen:
        log(f"chosen slot-series without usable pixel spacing (not physically cropped): "
            f"{n_unscaled}/{n_chosen} ({n_unscaled / n_chosen:.1%})")

    if DECODE_FAILED:
        reasons = Counter()
        for entry in DECODE_FAILED:
            for e in entry["errors"]:
                reasons[e.split(":")[0]] += 1
        log(f"decode failures: {len(DECODE_FAILED)} series affected")
        for reason, n in reasons.most_common(10):
            log(f"  {reason}: {n}")
    else:
        log("decode failures: none")


_CURRENT_LAT = {}


def process_split(root, split, out_dir, n_slice_target, order_cache_path):
    log(f"walking {split} series directories + reading headers")
    h = walk(root, split)
    if h.empty:
        log(f"{split}: no series found, skipping")
        return None
    h = annotate(h)

    plane_map = h.set_index("SeriesInstanceUID")["Anatomical_Plane"].to_dict()\
        if "Anatomical_Plane" in h.columns else {}
    if not plane_map:
        series_csv = root / f"{split}_series.csv"
        if series_csv.is_file():
            series_meta = pd.read_csv(series_csv)
            plane_map = series_meta.set_index("SeriesInstanceUID")["Anatomical_Plane"].to_dict()

    lat_map = lat_of(h, tag=f"{split} ")
    global _CURRENT_LAT
    _CURRENT_LAT = lat_map
    slot_map = pick_slots(h, plane_map)

    studies, cache, mask = build_cache(
        slot_map, cache_slices=n_slice_target, img_size=CACHE_IMG, tag=split,
        out_dir=out_dir, order_cache_path=order_cache_path,
    )

    pd.Series(studies, name="StudyInstanceUID").to_csv(
        Path(out_dir) / f"{split}_study_order.csv", index=False)
    coverage_report(split, studies, cache, mask, slot_map, lat_map)
    return studies, cache, mask


if __name__ == "__main__":
    ROOT = find_root()
    log(f"input root: {ROOT}")
    OUT_DIR.mkdir(exist_ok=True)

    train_df = pd.read_csv(ROOT / "train.csv")
    n_slice_target = plan_cache(len(train_df))

    process_split(ROOT, "train", OUT_DIR, n_slice_target, ORDER_CACHE_PATH)
    process_split(ROOT, "test", OUT_DIR, n_slice_target, ORDER_CACHE_PATH)

[    0.0s] input root: /kaggle/input/competitions/rsna-knee-abnormality-detection
[    0.1s] memory: 29.9 GB available, 13.5 GB budgeted for the cache -> 3 slices/slot
[    0.1s] walking train series directories + reading headers
[   79.5s] train laterality: 2203 from the tag, 2132 from geometry, 72 unresolved; tag and geometry disagree on 31 (1.4% of the tagged)
[  100.0s] creating new train_cache.dat (8.34 GB)
[  100.0s] train: cache (4407, 6, 3, 336, 336) = 8.3 GB
[  100.3s] train: 0 slot-series ordered from cache, 20130 to read (678385 slice headers)
[  115.3s]   train ordering 172/20130 (11/s, ~29min remaining)
[  130.6s]   train ordering 339/20130 (11/s, ~30min remaining)
[  145.9s]   train ordering 504/20130 (11/s, ~30min remaining)
[  161.1s]   train ordering 681/20130 (11/s, ~29min remaining)
[  176.2s]   train ordering 810/20130 (11/s, ~30min remaining)
[  191.2s]   train ordering 973/20130 (11/s, ~30min remaining)
[  206.3s]   train ordering 1145/20130 (11/s, ~29min remainin

In [3]:
%%writefile dataset.py
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

REGEX_LABEL_ACCURACY = {
    "ACL": 0.897, "MCL": 0.879, "Medial Meniscus": 0.776, "Lateral Meniscus": 0.862,
    "Medial OA": 0.845, "Lateral OA": 0.793, "PF OA": 0.741, "Effusion": 0.707,
    "Synovitis": 0.724, "Baker's": 0.879, "Contusion": 0.776, "Fracture": 0.810,
}


def load_manifest(preprocessed_dir):
    path = Path(preprocessed_dir) / "_MANIFEST.json"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found - run image_preprocessing.py (with the save-verification "
            f"guard rails) first, as a committed Kaggle version, before training.")
    return json.loads(path.read_text())


class KneeMRICache:
    def __init__(self, preprocessed_dir, split, manifest=None):
        preprocessed_dir = Path(preprocessed_dir)
        manifest = manifest or load_manifest(preprocessed_dir)
        info = manifest["splits"].get(split)
        if info is None:
            raise KeyError(f"no {split!r} entry in manifest; splits present: "
                           f"{list(manifest['splits'])}")
        shape = tuple(info["cache_shape"])
        self.cache = np.memmap(preprocessed_dir / f"{split}_cache.dat",
                               dtype=np.uint8, mode="r", shape=shape)
        self.mask = np.load(preprocessed_dir / f"{split}_mask.npy")
        self.study_order = pd.read_csv(
            preprocessed_dir / f"{split}_study_order.csv")["StudyInstanceUID"].tolist()
        self.study_to_idx = {s: i for i, s in enumerate(self.study_order)}
        if self.mask.shape[0] != len(self.study_order):
            raise ValueError(f"{split}: mask has {self.mask.shape[0]} rows but "
                             f"study_order has {len(self.study_order)} entries")


class KneeMRIDataset(Dataset):
    def __init__(self, cache: KneeMRICache, study_uids, labels_df=None):
        self.cache = cache
        self.study_uids = list(study_uids)
        self.labels_df = None
        self.weights = None
        if labels_df is not None:
            self.labels_df = labels_df.set_index("StudyInstanceUID")
            reg_acc = np.array([REGEX_LABEL_ACCURACY[t] for t in TARGETS], dtype=np.float32)
            self._silver_weight = reg_acc
            self._gold_weight = np.ones(len(TARGETS), dtype=np.float32)

    def __len__(self):
        return len(self.study_uids)

    def __getitem__(self, i):
        uid = self.study_uids[i]
        idx = self.cache.study_to_idx[uid]
        slots = torch.from_numpy(np.asarray(self.cache.cache[idx]))
        mask = torch.from_numpy(self.cache.mask[idx].copy())

        if self.labels_df is None:
            return uid, slots, mask

        row = self.labels_df.loc[uid]
        labels = torch.tensor([float(row[t]) for t in TARGETS], dtype=torch.float32)
        is_gold = str(row.get("label_source", "")).lower() == "gold"
        weight = self._gold_weight if is_gold else self._silver_weight
        weight = torch.from_numpy(weight.copy())
        return uid, slots, mask, labels, weight


def build_splits(preprocessed_dir, labels_csv, val_gold_frac=0.3, val_silver_frac=0.1,
                 seed=2026):
    labels_df = pd.read_csv(labels_csv)
    manifest = load_manifest(preprocessed_dir)
    cache_study_set = set(pd.read_csv(
        Path(preprocessed_dir) / "train_study_order.csv")["StudyInstanceUID"])
    labels_df = labels_df[labels_df["StudyInstanceUID"].isin(cache_study_set)].copy()

    rng = np.random.RandomState(seed)
    gold = labels_df[labels_df["label_source"] == "gold"]["StudyInstanceUID"].tolist()
    silver = labels_df[labels_df["label_source"] != "gold"]["StudyInstanceUID"].tolist()
    rng.shuffle(gold)
    rng.shuffle(silver)

    n_val_gold = max(1, int(round(len(gold) * val_gold_frac))) if gold else 0
    n_val_silver = int(round(len(silver) * val_silver_frac))

    val_gold_uids = gold[:n_val_gold]
    train_gold_uids = gold[n_val_gold:]
    val_silver_uids = silver[:n_val_silver]
    train_silver_uids = silver[n_val_silver:]

    train_uids = train_gold_uids + train_silver_uids
    rng.shuffle(train_uids)

    return train_uids, val_gold_uids, val_silver_uids, labels_df

Writing dataset.py


In [4]:
%%writefile model.py
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

DINOV2_CONFIGS = {
    "small": dict(hidden_size=384, num_hidden_layers=12, num_attention_heads=6, mlp_ratio=4),
    "base": dict(hidden_size=768, num_hidden_layers=12, num_attention_heads=12, mlp_ratio=4),
    "large": dict(hidden_size=1024, num_hidden_layers=24, num_attention_heads=16, mlp_ratio=4),
}


class ResNet50SlotEncoder(nn.Module):
    def __init__(self, out_dim=512, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = torchvision.models.resnet50(weights=weights)
        self.body = nn.Sequential(*list(backbone.children())[:-1])
        self.proj = nn.Linear(2048, out_dim)

    def forward(self, x):
        feat = self.body(x).flatten(1)
        return self.proj(feat)


def find_dinov2_checkpoint(variant="small", search_root="/kaggle/input"):
    root = Path(search_root)
    if not root.is_dir():
        return None
    hits = []
    for path in root.rglob("config.json"):
        if "dinov2" in str(path.parent).lower():
            hits.append(path.parent)
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None


class Dinov2SlotEncoder(nn.Module):
    def __init__(self, out_dim=512, pretrained=True, variant="small", unfreeze_last=2,
                source=None):
        super().__init__()
        from transformers import AutoModel, Dinov2Config, Dinov2Model

        if pretrained:
            path = source or find_dinov2_checkpoint(variant)
            if path is None:
                raise FileNotFoundError(
                    f"pretrained=True but no local DINOv2 '{variant}' checkpoint found "
                    f"under /kaggle/input - attach a DINOv2 model dataset to the "
                    f"notebook, or pass source= explicitly, or use pretrained=False.")
            self.backbone = AutoModel.from_pretrained(str(path))
        else:
            cfg = Dinov2Config(**DINOV2_CONFIGS[variant])
            self.backbone = Dinov2Model(cfg)

        n_layers = len(self.backbone.encoder.layer)
        for p in self.backbone.parameters():
            p.requires_grad = False
        for blk in self.backbone.encoder.layer[max(0, n_layers - unfreeze_last):]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in self.backbone.layernorm.parameters():
            p.requires_grad = True

        hidden_size = self.backbone.config.hidden_size
        self.proj = nn.Linear(hidden_size, out_dim)

    def forward(self, x):
        out = self.backbone(pixel_values=x, interpolate_pos_encoding=True)
        cls_token = out.last_hidden_state[:, 0]
        return self.proj(cls_token)


def build_encoder(backbone, out_dim=512, pretrained=True, **kwargs):
    if backbone == "resnet50":
        return ResNet50SlotEncoder(out_dim=out_dim, pretrained=pretrained)
    if backbone == "dinov2":
        return Dinov2SlotEncoder(out_dim=out_dim, pretrained=pretrained, **kwargs)
    raise ValueError(f"unknown backbone {backbone!r}, expected 'resnet50' or 'dinov2'")


class KneeMRIModel(nn.Module):
    def __init__(self, n_labels=12, feat_dim=512, backbone="resnet50", pretrained=True,
                train_img_size=224, backbone_kwargs=None):
        super().__init__()
        self.encoder = build_encoder(backbone, out_dim=feat_dim, pretrained=pretrained,
                                     **(backbone_kwargs or {}))
        self.backbone_name = backbone
        self.train_img_size = train_img_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim * 2),
            nn.Linear(feat_dim * 2, feat_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(feat_dim, n_labels),
        )
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, slots, mask):
        B, S, C, H, W = slots.shape
        x = slots.reshape(B * S, C, H, W).float() / 255.0
        if x.shape[-1] != self.train_img_size:
            x = F.interpolate(x, size=(self.train_img_size, self.train_img_size),
                              mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std

        feat = self.encoder(x).reshape(B, S, -1)

        mask_f = mask.unsqueeze(-1)
        mean_feat = (feat * mask_f).sum(1) / mask_f.sum(1).clamp(min=1.0)

        neg_inf = torch.finfo(feat.dtype).min
        masked_for_max = feat.masked_fill(mask_f == 0, neg_inf)
        max_feat = masked_for_max.max(1).values
        no_slots = (mask.sum(1) == 0).unsqueeze(-1)
        max_feat = torch.where(no_slots, torch.zeros_like(max_feat), max_feat)

        pooled = torch.cat([mean_feat, max_feat], dim=-1)
        return self.classifier(pooled)

Writing model.py


In [5]:
%%writefile train.py
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path(__file__).parent))
from dataset import KneeMRICache, KneeMRIDataset, build_splits, TARGETS
from model import KneeMRIModel

DEFAULT_PREPROCESSED_DIR = "preprocessed"
DEFAULT_LABELS_CSV = "report_labels_v1_regex.csv"
OUT_DIR = Path("training_output")
TIME_BUDGET_S = 8.5 * 3600
BATCH_SIZE = 16
NUM_WORKERS = 4
BACKBONE_LR = 1e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 40
PATIENCE = 6
WARMUP_EPOCHS = 2
GRAD_CLIP = 5.0


def warn_if_interactive():
    run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
    if run_type and run_type.lower() != "batch":
        print(f"[WARNING] KAGGLE_KERNEL_RUN_TYPE={run_type!r} - this looks like an "
              f"interactive Draft Session. /kaggle/working here is EPHEMERAL and will "
              f"not persist. Use 'Save Version -> Save & Run All (Commit)' to get a "
              f"run whose output actually survives.", flush=True)


def collate_labeled(batch):
    uids, slots, mask, labels, weight = zip(*batch)
    return (list(uids), torch.stack(slots), torch.stack(mask),
            torch.stack(labels), torch.stack(weight))


def make_loader(cache, uids, labels_df, batch_size, shuffle):
    ds = KneeMRIDataset(cache, uids, labels_df)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      collate_fn=collate_labeled, drop_last=False)


def compute_pos_weight(labels_df, uids):
    sub = labels_df.set_index("StudyInstanceUID").loc[uids]
    pos = sub[TARGETS].sum(axis=0).values.astype(np.float32)
    neg = len(uids) - pos
    pos = np.clip(pos, 1, None)
    return torch.from_numpy(neg / pos)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    for uids, slots, mask, labels, _ in loader:
        slots, mask = slots.to(device), mask.to(device)
        logits = model(slots, mask)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.numpy())
    if not all_logits:
        return float("nan"), {}
    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    per_label_auc = {}
    for i, t in enumerate(TARGETS):
        y = labels[:, i]
        if len(np.unique(y)) < 2:
            continue
        try:
            per_label_auc[t] = roc_auc_score(y, logits[:, i])
        except ValueError:
            continue
    mean_auc = float(np.mean(list(per_label_auc.values()))) if per_label_auc else float("nan")
    return mean_auc, per_label_auc


def save_checkpoint(path, model, optimizer, scaler, epoch, best_auc, rng_state):
    tmp = Path(str(path) + ".tmp")
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "best_auc": best_auc,
        "torch_rng_state": rng_state,
    }, tmp)
    os.replace(tmp, path)


def load_checkpoint(path, model, optimizer, scaler, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    torch.set_rng_state(ckpt["torch_rng_state"])
    return ckpt["epoch"], ckpt["best_auc"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--preprocessed-dir", default=DEFAULT_PREPROCESSED_DIR)
    ap.add_argument("--labels-csv", default=DEFAULT_LABELS_CSV)
    ap.add_argument("--out-dir", default=str(OUT_DIR))
    ap.add_argument("--batch-size", type=int, default=BATCH_SIZE)
    ap.add_argument("--max-epochs", type=int, default=MAX_EPOCHS)
    ap.add_argument("--time-budget-s", type=float, default=TIME_BUDGET_S)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--pretrained", action="store_true", default=True)
    ap.add_argument("--no-pretrained", dest="pretrained", action="store_false")
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--backbone", choices=["resnet50", "dinov2"], default="resnet50")
    ap.add_argument("--dinov2-variant", choices=["small", "base", "large"], default="small")
    ap.add_argument("--dinov2-source", default=None,
                    help="path to a local DINOv2 HF checkpoint dir; if unset, "
                         "auto-searched under /kaggle/input")
    ap.add_argument("--dinov2-unfreeze-last", type=int, default=2,
                    help="number of trailing transformer blocks to fine-tune; "
                         "the rest of the backbone stays frozen")
    args = ap.parse_args()

    warn_if_interactive()
    start_time = time.time()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "train_log.jsonl"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] device={device}", flush=True)

    train_uids, val_gold_uids, val_silver_uids, labels_df = build_splits(
        args.preprocessed_dir, args.labels_csv)
    print(f"[INFO] train={len(train_uids)} val_gold={len(val_gold_uids)} "
          f"val_silver={len(val_silver_uids)}", flush=True)
    if len(val_gold_uids) == 0:
        print("[WARNING] no gold studies available for validation - checkpoint "
              "selection will fall back to silver AUC, which is a noisier signal "
              "since silver labels are regex-derived, not radiologist-derived.",
              flush=True)

    cache = KneeMRICache(args.preprocessed_dir, "train")

    train_loader = make_loader(cache, train_uids, labels_df, args.batch_size, shuffle=True)
    val_gold_loader = make_loader(cache, val_gold_uids, labels_df, args.batch_size, shuffle=False) \
        if val_gold_uids else None
    val_silver_loader = make_loader(cache, val_silver_uids, labels_df, args.batch_size, shuffle=False) \
        if val_silver_uids else None

    pos_weight = compute_pos_weight(labels_df, train_uids).to(device)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")

    backbone_kwargs = {}
    if args.backbone == "dinov2":
        backbone_kwargs = {
            "variant": args.dinov2_variant,
            "unfreeze_last": args.dinov2_unfreeze_last,
            "source": args.dinov2_source,
        }
    model = KneeMRIModel(
        n_labels=len(TARGETS),
        backbone=args.backbone,
        pretrained=args.pretrained,
        train_img_size=args.img_size,
        backbone_kwargs=backbone_kwargs,
    ).to(device)

    pretrained_submodule = (
        model.encoder.body if args.backbone == "resnet50" else model.encoder.backbone)
    backbone_params = [p for p in pretrained_submodule.parameters() if p.requires_grad]
    head_params = list(model.encoder.proj.parameters()) + list(model.classifier.parameters())
    n_backbone_trainable = sum(p.numel() for p in backbone_params)
    n_head_trainable = sum(p.numel() for p in head_params)
    print(f"[INFO] backbone={args.backbone} backbone_trainable_params="
          f"{n_backbone_trainable/1e6:.1f}M head_trainable_params={n_head_trainable/1e6:.1f}M",
          flush=True)
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, args.max_epochs - WARMUP_EPOCHS))
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    start_epoch = 0
    best_auc = -1.0
    ckpt_path = out_dir / "last.pt"
    best_path = out_dir / "best.pt"
    if args.resume and ckpt_path.is_file():
        start_epoch, best_auc = load_checkpoint(ckpt_path, model, optimizer, scaler, device)
        start_epoch += 1
        print(f"[INFO] resumed from epoch {start_epoch}, best_auc so far={best_auc:.4f}",
              flush=True)

    epochs_since_improve = 0
    stopped_reason = None

    for epoch in range(start_epoch, args.max_epochs):
        if time.time() - start_time > args.time_budget_s:
            stopped_reason = "time_budget"
            print(f"[INFO] time budget ({args.time_budget_s}s) reached before epoch "
                  f"{epoch}, stopping gracefully", flush=True)
            break

        if epoch < WARMUP_EPOCHS:
            warmup_scale = (epoch + 1) / WARMUP_EPOCHS
            for g, base_lr in zip(optimizer.param_groups, [BACKBONE_LR, HEAD_LR]):
                g["lr"] = base_lr * warmup_scale

        model.train()
        running_loss, n_batches = 0.0, 0
        for uids, slots, mask, labels, weight in train_loader:
            slots, mask = slots.to(device), mask.to(device)
            labels, weight = labels.to(device), weight.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(slots, mask)
                per_elem = bce(logits, labels)
                loss = (per_elem * weight).sum() / weight.sum().clamp(min=1.0)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            n_batches += 1

        if epoch >= WARMUP_EPOCHS:
            scheduler.step()

        train_loss = running_loss / max(1, n_batches)
        gold_auc, gold_per_label = (float("nan"), {}) if val_gold_loader is None \
            else evaluate(model, val_gold_loader, device)
        silver_auc, _ = (float("nan"), {}) if val_silver_loader is None \
            else evaluate(model, val_silver_loader, device)

        selection_auc = gold_auc if val_gold_loader is not None else silver_auc

        elapsed = time.time() - start_time
        print(f"[epoch {epoch}] loss={train_loss:.4f} gold_auc={gold_auc:.4f} "
              f"silver_auc={silver_auc:.4f} elapsed={elapsed/60:.1f}min", flush=True)

        with open(log_path, "a") as f:
            f.write(json.dumps({
                "epoch": epoch, "train_loss": train_loss, "gold_auc": gold_auc,
                "silver_auc": silver_auc, "gold_per_label_auc": gold_per_label,
                "elapsed_s": elapsed,
            }) + "\n")

        rng_state = torch.get_rng_state()
        save_checkpoint(ckpt_path, model, optimizer, scaler, epoch, best_auc, rng_state)

        if not np.isnan(selection_auc) and selection_auc > best_auc:
            best_auc = selection_auc
            epochs_since_improve = 0
            save_checkpoint(best_path, model, optimizer, scaler, epoch, best_auc, rng_state)
            print(f"[epoch {epoch}] new best (auc={best_auc:.4f}) -> saved {best_path}", flush=True)
        else:
            epochs_since_improve += 1

        if epochs_since_improve >= PATIENCE:
            stopped_reason = "early_stop"
            print(f"[INFO] no improvement for {PATIENCE} epochs, stopping", flush=True)
            break
    else:
        stopped_reason = "max_epochs"

    if not best_path.is_file():
        raise RuntimeError(
            "training finished without ever saving best.pt - this means validation "
            "AUC was NaN every epoch (no usable val labels), which means the run "
            "produced no usable model. Check that val_gold_uids / labels_df are "
            "non-empty before trusting any downstream inference.")

    summary = {"stopped_reason": stopped_reason, "best_auc": best_auc,
              "final_epoch": epoch, "elapsed_s": time.time() - start_time}
    (out_dir / "_TRAIN_MANIFEST.json").write_text(json.dumps(summary, indent=2))
    print(f"[DONE] {summary}", flush=True)


if __name__ == "__main__":
    main()

Writing train.py


In [6]:
!python train.py \
  --preprocessed-dir preprocessed \
  --labels-csv report_labels_v1_regex.csv \
  --out-dir training_output \
  --backbone resnet50

[INFO] device=cuda
Traceback (most recent call last):
  File "/kaggle/working/train.py", line 299, in <module>
    main()
  File "/kaggle/working/train.py", line 145, in main
    train_uids, val_gold_uids, val_silver_uids, labels_df = build_splits(
                                                            ^^^^^^^^^^^^^
  File "/kaggle/working/dataset.py", line 88, in build_splits
    manifest = load_manifest(preprocessed_dir)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/dataset.py", line 27, in load_manifest
    raise FileNotFoundError(
FileNotFoundError: preprocessed/_MANIFEST.json not found - run image_preprocessing.py (with the save-verification guard rails) first, as a committed Kaggle version, before training.
